# E2 · Ablation Table 4 trên CIFAR-100
Bật Internet, chọn GPU, gắn dataset backbone ba seed và cấp quyền cho secret `WANDB_API_KEY`. Notebook clone nhánh `feat/e2-table4-ablation` từ GitHub và chạy test trước khi đánh giá. Trong run W&B `E2-table4-comparison`, mở bảng `table4` để xem 5 biến thể cho từng seed 1, 2 và 3.

E2 dùng lớp LegacyFeCAM trong notebook để tái hiện phép chấm điểm cũ: prototype và query giữ độ dài sau Tukey, ma trận precision được tính bằng pseudoinverse. Backbone, loader và NCM được import từ nhánh GitHub.


In [ ]:
import hashlib
import importlib.util
import os
from pathlib import Path
import subprocess
import sys
import tempfile

BRANCH = "feat/e2-table4-ablation"
REMOTE = "https://github.com/duongthienphu/fecam-study.git"
WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
WORK.mkdir(parents=True, exist_ok=True)
REPO = Path(tempfile.mkdtemp(prefix="e2-code-", dir=WORK))
subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
                REMOTE, str(REPO)], check=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
source_sha = hashlib.sha256((REPO / "src/models/fecam.py").read_bytes()).hexdigest()
print("Source:", REPO, "FeCAM SHA256:", source_sha)
os.chdir(WORK)
sys.path.insert(0, str(REPO))
missing = [pkg for mod, pkg in [("datasets", "datasets"), ("wandb", "wandb"), ("yaml", "PyYAML")]
           if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)


In [ ]:
from dataclasses import dataclass
import math

import torch
from torch import Tensor


@dataclass(frozen=True)
class _LegacyFeCAMConfig:
    transform: str
    gamma1: float
    gamma2: float
    correlation_normalization: bool


class LegacyFeCAM:
    """Per-class covariance classifier with atomic incremental updates.

    Statistics and returned distances use float64 on ``device`` (CPU default).
    Returned labels/class_ids use int64 on that same device. Inputs are detached
    and moved there automatically. Only new classes are accepted by update.
    Configuration is fixed at construction so old/new statistics stay compatible.
    """

    def __init__(
        self,
        *,
        transform: str = "sqrt",
        gamma1: float = 1.0,
        gamma2: float = 1.0,
        correlation_normalization: bool = True,
        device: str | torch.device = "cpu",
    ) -> None:
        if transform not in ("none", "sqrt"):
            raise ValueError("transform must be 'none' or 'sqrt'.")
        gamma1, gamma2 = float(gamma1), float(gamma2)
        if any(not math.isfinite(g) or g < 0 for g in (gamma1, gamma2)):
            raise ValueError("gamma1 and gamma2 must be finite and nonnegative.")
        self._config = _LegacyFeCAMConfig(transform, gamma1, gamma2, correlation_normalization)
        self._device = torch.device(device)
        self._d: int | None = None
        self.class_ids = torch.empty(0, dtype=torch.int64, device=self._device)
        self.counts = torch.empty(0, dtype=torch.int64, device=self._device)
        self.raw_prototypes = torch.empty((0, 0), dtype=torch.float64, device=self._device)
        self.score_prototypes = self.raw_prototypes.clone()
        self.precisions = torch.empty((0, 0, 0), dtype=torch.float64, device=self._device)

    @property
    def num_classes(self) -> int:
        return self.class_ids.numel()

    @property
    def feature_dim(self) -> int | None:
        return self._d

    @property
    def id_to_column(self) -> dict[int, int]:
        return {label: column for column, label in enumerate(self.class_ids.tolist())}

    def _features(self, features: Tensor, *, allow_empty: bool) -> Tensor:
        if not isinstance(features, Tensor) or not features.is_floating_point():
            raise TypeError("features must be a floating-point torch.Tensor.")
        if features.ndim != 2 or features.shape[1] == 0:
            raise ValueError("features must have shape [N, d] with d > 0.")
        if not allow_empty and features.shape[0] == 0:
            raise ValueError("update requires at least one sample.")
        if self._d is not None and features.shape[1] != self._d:
            raise ValueError(f"Expected {self._d} features, got {features.shape[1]}.")
        h = features.detach().to(device=self._device, dtype=torch.float64)
        if not torch.isfinite(h).all().item():
            raise ValueError("features contain NaN or infinity.")
        if self._config.transform == "sqrt" and (h < 0).any().item():
            raise ValueError("sqrt Tukey requires nonnegative features; use 'none' for signed features.")
        return h

    def _transform(self, h: Tensor) -> Tensor:
        return h.sqrt() if self._config.transform == "sqrt" else h

    def _build_precision(self, covariance: Tensor, class_id: int) -> Tensor:
        """Shrink, optionally correlation normalize, then compute the pseudoinverse."""
        d = covariance.shape[0]
        covariance = (covariance + covariance.T) / 2
        eye = torch.eye(d, dtype=covariance.dtype, device=covariance.device)
        diagonal = covariance.diagonal()
        v1 = diagonal.mean()
        v2 = ((covariance.sum() - diagonal.sum()) / (d * (d - 1))
              if d > 1 else covariance.new_zeros(()))
        shrunk = (covariance + self._config.gamma1 * v1 * eye
                  + self._config.gamma2 * v2 * (torch.ones_like(eye) - eye))
        if not torch.isfinite(shrunk).all().item():
            raise ValueError(f"Class {class_id}: nonfinite covariance.")
        if self._config.correlation_normalization:
            if (shrunk.diagonal() <= 0).any().item():
                raise ValueError(f"Class {class_id}: normalization requires positive variances.")
            std = shrunk.diagonal().sqrt()
            shrunk = shrunk / torch.outer(std, std)
        if not torch.isfinite(shrunk).all().item():
            raise ValueError(f"Class {class_id}: normalization overflowed.")
        return torch.linalg.pinv((shrunk + shrunk.T) / 2, hermitian=True)

    @torch.no_grad()
    def update(self, features: Tensor, labels: Tensor) -> None:
        """Add complete new classes; preserve old statistics on success or failure.

        Each class must have >= 2 training samples. Repeated class IDs are rejected.
        New IDs are sorted within this call and appended after existing IDs.
        """
        h = self._features(features, allow_empty=False)
        if not isinstance(labels, Tensor) or labels.dtype != torch.int64:
            raise TypeError("labels must be a torch.int64 tensor of global class IDs.")
        if labels.ndim != 1 or labels.shape[0] != h.shape[0]:
            raise ValueError("labels must have shape [N], matching features.")
        labels = labels.detach().to(self._device)
        new_ids = torch.unique(labels, sorted=True)
        overlap = set(new_ids.tolist()).intersection(self.class_ids.tolist())
        if overlap:
            raise ValueError(f"Classes already learned: {sorted(overlap)}. Pass only new classes.")

        raw_means, score_means, precisions, counts = [], [], [], []
        for class_id in new_ids.tolist():
            h_y = h[labels == class_id]
            n = h_y.shape[0]
            if n < 2:
                raise ValueError(f"Class {class_id}: need at least 2 samples, got {n}.")
            mu = h_y.mean(dim=0)
            z_y = self._transform(h_y)
            centered = z_y - z_y.mean(dim=0)
            covariance = centered.T @ centered / (n - 1)
            precision = self._build_precision(covariance, class_id)
            score_mu = self._transform(mu)
            if not torch.isfinite(score_mu).all().item():
                raise ValueError(f"Class {class_id}: prototype overflowed.")
            raw_means.append(mu)
            score_means.append(score_mu)
            precisions.append(precision)
            counts.append(n)

        # Build every new state tensor BEFORE assigning any persistent attribute.
        def append(old: Tensor, new: Tensor) -> Tensor:
            return torch.cat((old, new), dim=0) if self.num_classes else new

        state = (
            append(self.class_ids, new_ids),
            append(self.raw_prototypes, torch.stack(raw_means)),
            append(self.score_prototypes, torch.stack(score_means)),
            append(self.precisions, torch.stack(precisions)),
            append(self.counts, torch.tensor(counts, dtype=torch.int64, device=self._device)),
        )
        (self.class_ids, self.raw_prototypes, self.score_prototypes,
         self.precisions, self.counts) = state
        self._d = h.shape[1]

    @torch.no_grad()
    def distances(self, features: Tensor) -> Tensor:
        """Return squared Mahalanobis distances [B, C]; smaller is better.

        Column j corresponds to class_ids[j]. These values are not probabilities.
        Query batches may be empty. Batch large query sets in the caller.
        """
        if not self.num_classes:
            raise RuntimeError("Call update before prediction.")
        z = self._transform(self._features(features, allow_empty=True))
        result = z.new_empty((z.shape[0], self.num_classes))
        if z.shape[0] == 0:
            return result
        for column in range(self.num_classes):
            residual = z - self.score_prototypes[column]
            result[:, column] = ((residual @ self.precisions[column]) * residual).sum(dim=1)
        if not torch.isfinite(result).all().item():
            raise FloatingPointError("Nonfinite distances; check feature scale and conditioning.")
        return result

    @torch.no_grad()
    def predict(self, features: Tensor) -> Tensor:
        """Return global class IDs [B]; ties select the first stored class."""
        distances = self.distances(features)
        if distances.shape[0] == 0:
            return self.class_ids.new_empty((0,))
        return self.class_ids[distances.argmin(dim=1)]

In [ ]:
"""Classifier settings and reference percentages for the CIFAR-100 Table 4 experiment."""

from dataclasses import dataclass
from statistics import mean

import numpy as np
import torch
from src.models.ncm import NCMClassifier


class E2NCM(NCMClassifier):
    """Use the legacy runner's Euclidean NCM and optional sqrt(mean(raw))."""

    def __init__(self, transform):
        super().__init__(metric="euclidean", normalize=False)
        self.transform = transform

    def update(self, features, labels):
        self.update_prototypes(features.cpu().numpy(), labels.cpu().numpy())
        if self.transform == "sqrt":
            for label in labels.unique().tolist():
                self.prototypes[label] = np.sqrt(self.prototypes[label])

    def predict(self, batch):
        features = batch.sqrt() if self.transform == "sqrt" else batch
        return torch.from_numpy(super().predict(features.cpu().numpy()))

    @property
    def num_classes(self):
        return len(self.prototypes)



@dataclass(frozen=True)
class Variant:
    covariance: str
    transform: str
    correlation: bool
    paper_avg: float
    paper_final: float

    def classifier(self, device):
        if self.covariance == "none":
            return E2NCM(self.transform)
        return LegacyFeCAM(transform=self.transform,
                     correlation_normalization=self.correlation, device=device)


VARIANTS = {
    "ncm": Variant("none", "none", False, 64.8, 51.6),
    "ncm-tukey": Variant("none", "sqrt", False, 66.6, 54.4),
    "fecam-full": Variant("classwise", "sqrt", True, 70.9, 62.1),
    "fecam-no-tukey": Variant("classwise", "none", True, 65.9, 55.4),
    "fecam-no-norm": Variant("classwise", "sqrt", False, 62.8, 52.1),
}


def summarize(name, seed, sessions):
    if [row["num_seen_classes"] for row in sessions] != [50, 60, 70, 80, 90, 100]:
        raise ValueError("Expected all six B50-I10 sessions in order")
    scores = [row["seen_acc_pct"] for row in sessions]
    if not all(0 <= score <= 100 for score in scores):
        raise ValueError("Expected percentage accuracies")
    setting = VARIANTS[name]
    avg, final = mean(scores), scores[-1]
    return dict(run=f"{name}__seed{seed}", seed=seed, variant=name,
                avg_acc_pct=avg, final_acc_pct=final,
                paper_avg_acc_pct=setting.paper_avg, paper_final_acc_pct=setting.paper_final,
                avg_delta_pp=avg - setting.paper_avg, final_delta_pp=final - setting.paper_final)


In [ ]:
import unittest

import torch




class TestTable4(unittest.TestCase):
    def test_component_settings(self):
        settings = [(v.covariance, v.transform, v.correlation) for v in VARIANTS.values()]
        self.assertEqual(settings, [
            ("none", "none", False), ("none", "sqrt", False), ("classwise", "sqrt", True),
            ("classwise", "none", True), ("classwise", "sqrt", False),
        ])

    def test_incremental_predictions(self):
        base = torch.tensor([[4., 0.1], [3.8, 0.2], [4.1, 0.3]])
        for name, variant in VARIANTS.items():
            with self.subTest(name=name):
                model = variant.classifier("cpu")
                model.update(base, torch.zeros(3, dtype=torch.long))
                model.update(base.flip([1]), torch.ones(3, dtype=torch.long))
                self.assertEqual(model.predict(torch.stack([base[0], base[0].flip([0])])).tolist(),
                                 [0, 1])

    def test_legacy_ncm_uses_magnitude_and_transform_of_mean(self):
        x = torch.tensor([[1., 0.], [3., 0.], [7., 0.], [9., 0.]])
        y = torch.tensor([0, 0, 1, 1])
        for name in ("ncm", "ncm-tukey"):
            model = VARIANTS[name].classifier("cpu")
            model.update(x, y)
            self.assertFalse(model.normalize)
            self.assertEqual(model.predict(x[[0, 3]]).tolist(), [0, 1])
            self.assertAlmostEqual(float(model.prototypes[0][0]),
                                   2 ** 0.5 if name == "ncm-tukey" else 2., places=6)

    def test_metrics(self):
        sessions = [dict(num_seen_classes=n, seen_acc_pct=130-n) for n in range(50, 101, 10)]
        row = summarize("ncm", 2, sessions)
        self.assertEqual((row["avg_acc_pct"], row["final_acc_pct"]), (55, 30))
        self.assertAlmostEqual(row["avg_delta_pp"], -9.8)
        with self.assertRaises(ValueError):
            summarize("ncm", 2, sessions[:-1])

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestTable4)
if not unittest.TextTestRunner(verbosity=2).run(suite).wasSuccessful():
    raise RuntimeError("E2 tests failed")


In [ ]:
import json
import pandas as pd
import torch
import wandb
from kaggle_secrets import UserSecretsClient
from scripts.run_experiment import (checkpoint_sha256, extract_features, load_checkpoint,
                                    load_cifar100, load_config, make_loader, seed_everything,
                                    TaskSplit, validate_checkpoint)

PROJECT = "fecam-study"
ENTITY = "duongthienphu-university-of-science-vnuhcm"
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login()
device = "cuda" if torch.cuda.is_available() else "cpu"
seeds = (1, 2, 3)
paths = {}
for seed in seeds:
    matches = sorted(INPUT.rglob(f"backbone-seed{seed}/base.pt"))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one backbone-seed{seed}/base.pt, found {matches}")
    paths[seed] = matches[0]
out = Path(tempfile.mkdtemp(prefix="e2-results-", dir=WORK))
data = load_cifar100(WORK / "cifar-data")
print(device, out)

In [ ]:
def evaluate_seed(seed):
    # E1 and E2 use the same frozen CIFAR-100 B50-I10 backbone protocol.
    cfg = load_config(REPO / "configs/experiments/e1_fecam_classwise.yaml")
    cfg["seed"] = seed
    seed_everything(seed)
    split = TaskSplit(**cfg["split"])
    backbone, metadata = load_checkpoint(paths[seed], device)
    validate_checkpoint(metadata, cfg, split)
    models = {name: spec.classifier(device) for name, spec in VARIANTS.items()}
    histories = {name: [] for name in models}
    test_features, test_labels = [], []
    for session in range(len(split.tasks)):
        train_loader = make_loader(data, split, cfg, session, "train", device)
        test_loader = make_loader(data, split, cfg, session, "test", device)
        train_x, train_y = extract_features(backbone, train_loader, device)
        test_x, test_y = extract_features(backbone, test_loader, device)
        test_features.append(test_x)
        test_labels.append(test_y)
        test_x, test_y = torch.cat(test_features), torch.cat(test_labels)
        for name, model in models.items():
            model.update(train_x, train_y)
            preds = torch.cat([model.predict(batch).cpu() for batch in test_x.split(256)])
            correct = (preds == test_y).sum().item()
            histories[name].append(dict(session=session, num_seen_classes=model.num_classes,
                                        seen_acc_pct=100 * correct / len(test_y)))
        print(f"seed={seed} session={session} classes={models['ncm'].num_classes}", flush=True)
    return histories, {"git_commit": commit, "repo_fecam_sha256": source_sha, "e2_scoring": "legacy_notebook", "seed": seed, "protocol": cfg["protocol"],
                       "class_order": split.class_order, "checkpoint_sha256": checkpoint_sha256(paths[seed])}


rows = []
for seed in seeds:
    histories, provenance = evaluate_seed(seed)
    for name, sessions in histories.items():
        row = summarize(name, seed, sessions)
        spec = VARIANTS[name]
        with wandb.init(project=PROJECT, entity=ENTITY, name=row["run"], group="E2", job_type="evaluation",
                        tags=["E2", "table4-ablation"], dir=str(out),
                        config={**provenance, "variant": name, "transform": spec.transform,
                                "covariance_mode": spec.covariance,
                                "normalize_features": False, "normalize_prototypes": False,
                                "covariance_inverse": "pinv" if spec.covariance == "classwise" else None,
                                "correlation_normalization": spec.correlation}) as run:
            run.define_metric("session")
            run.define_metric("seen_acc_pct", step_metric="session")
            for record in sessions:
                run.log(record, step=record["session"])
            run.summary.update({k: v for k, v in row.items() if k not in ("run", "seed", "variant")})
            run.summary.update({
                "avg_relative_delta_pct": 100 * row["avg_delta_pp"] / spec.paper_avg,
                "final_relative_delta_pct": 100 * row["final_delta_pp"] / spec.paper_final,
            })
            row["wandb_url"] = run.url
        rows.append(row)
    (out / f"seed{seed}-sessions.json").write_text(json.dumps(
        {"provenance": provenance, "sessions": histories}, indent=2))
    pd.DataFrame(rows).to_csv(out / "table4_runs.csv", index=False)

In [ ]:
runs = pd.DataFrame(rows)
expected = {(seed, variant) for seed in seeds for variant in VARIANTS}
if len(runs) != len(expected) or set(zip(runs["seed"], runs["variant"])) != expected:
    raise ValueError("Expected one run per seed and variant")
columns = ["run", "seed", "variant", "avg_acc_pct", "final_acc_pct",
           "paper_avg_acc_pct", "paper_final_acc_pct", "avg_delta_pp", "final_delta_pp"]
table = runs.loc[:, columns].copy()
table.to_csv(out / "table4_comparison.csv", index=False)
with wandb.init(project=PROJECT, entity=ENTITY, name="E2-table4-comparison", group="E2", job_type="summary",
                tags=["E2", "table4-ablation"], dir=str(out), config={"git_commit": commit, "repo_fecam_sha256": source_sha, "e2_scoring": "legacy_notebook"}) as run:
    run.log({"table4": wandb.Table(dataframe=table)})
    artifact = wandb.Artifact("e2-table4-results", type="evaluation")
    for path in sorted(out.glob("*")):
        if path.suffix in {".csv", ".json"}:
            artifact.add_file(str(path))
    run.log_artifact(artifact)
print(out)
display(table)